# Scripts for measuring noise on a large skypatch image

In [ ]:
# | default_exp euclid.skypatch_noise

In [ ]:
# | exporti

import logging
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from astropy.io import fits
from astropy.stats import sigma_clip
from astropy.visualization import simple_norm
from photutils.aperture import CircularAnnulus, EllipticalAnnulus
from scipy.stats import median_abs_deviation

In [ ]:
# | exporti

plt.style.use("nicl.euclid.v1nicl")

In [ ]:
# | export


def noise_profile_diagnostics(
    extended_measurement_table,
    label=None,
    output_path=None,
    zp=23.9,
):
    output_path = Path(output_path)
    grouped = extended_measurement_table.groupby("Centre_pixel")

    def plot_profiles(ax, x, y, ylabel):
        ax.plot(x, y, marker="o", markersize=2)
        ax.hlines(0, xmin=np.min(x), xmax=np.max(x), color="k", lw=1, ls="-")
        ax.set_ylabel(ylabel, fontsize=14)
        ax.set_xlabel("Radius (arcsec)", fontsize=14)
        ax.set_ylim(-0.2, 0.2)

    # Plot: Clipped median & mean flux profiles
    fig, ax = plt.subplots(1, 2, figsize=(8, 4))
    for _, group in grouped:
        plot_profiles(
            ax[0],
            group["SMA_annulus_centre_arcsec"],
            group["Clipped_median_flux_annulus"],
            "Clipped Median Flux",
        )
        plot_profiles(
            ax[1],
            group["SMA_annulus_centre_arcsec"],
            group["Clipped_mean_flux_annulus"],
            "Clipped Mean Flux",
        )
    fig.tight_layout()
    if output_path:
        plt.savefig(output_path / f"{label}_Median_Mean_Flux_CLIPPED.pdf")
        plt.close()
    else:
        plt.show()

    # Plot: Background-subtracted flux profiles
    fig, ax = plt.subplots(figsize=(4, 4))
    for _, group in grouped:
        ax.plot(
            group["SMA_annulus_centre_arcsec"],
            group["bkg_sub_flux"],
            marker="o",
            markersize=2,
        )
        ax.hlines(
            group["bkg_noise"].iloc[0],
            xmin=np.min(group["SMA_annulus_centre_arcsec"]),
            xmax=np.max(group["SMA_annulus_centre_arcsec"]),
            color="k",
            lw=0.2,
            ls="--",
        )
    ax.hlines(
        0,
        xmin=0,
        xmax=np.max(extended_measurement_table["SMA_annulus_centre_arcsec"]),
        color="k",
        lw=0.5,
        ls="-",
    )
    ax.set_ylabel("Background-subtracted Median Flux", fontsize=12)
    ax.set_xlabel("Radius (arcsec)", fontsize=12)
    fig.tight_layout()
    if output_path:
        plt.savefig(output_path / f"{label}_BkgsubFlux.pdf")
        plt.close()
    else:
        plt.show()

    # Plot: MAD profiles in flux and magnitude space
    flux_grouped_rad = extended_measurement_table.groupby("SMA_annulus_centre_arcsec")
    radii = sorted(flux_grouped_rad.groups.keys())

    mad_median_flux = [
        median_abs_deviation(
            flux_grouped_rad.get_group(r)["Median_flux_annulus"],
            scale="normal",
            nan_policy="omit",
        )
        for r in radii
    ]
    mad_clipped_median_flux = [
        median_abs_deviation(
            flux_grouped_rad.get_group(r)["Clipped_median_flux_annulus"],
            scale="normal",
            nan_policy="omit",
        )
        for r in radii
    ]
    mad_bkg_sub_flux = [
        median_abs_deviation(
            flux_grouped_rad.get_group(r)["bkg_sub_flux"],
            scale="normal",
            nan_policy="omit",
        )
        for r in radii
    ]

    fig, ax = plt.subplots(1, 2, figsize=(8, 4))

    # Plot in flux units
    ax[0].plot(radii, np.array(mad_median_flux) * 3, label="MAD of Median Flux")
    ax[0].plot(
        radii, np.array(mad_clipped_median_flux) * 3, label="MAD of Clipped Median Flux"
    )
    ax[0].plot(
        radii, np.array(mad_bkg_sub_flux) * 3, label="MAD of Bkg Subtracted Flux"
    )
    ax[0].set_ylabel("3xMAD (Flux / arcsec²)", fontsize=14)
    # ax[0].set_xscale("log")
    ax[0].grid(True)
    ax[0].legend()
    ax[0].set_ylim(-0.2, 2)

    # Plot in magnitude units
    def flux_to_mag(flux):
        return zp - 2.5 * np.log10(flux)

    ax[1].plot(
        radii, flux_to_mag(np.array(mad_median_flux) * 3), label="MAD of Median Flux"
    )
    ax[1].plot(
        radii,
        flux_to_mag(np.array(mad_clipped_median_flux) * 3),
        label="MAD of Clipped Median Flux",
    )
    ax[1].plot(
        radii,
        flux_to_mag(np.array(mad_bkg_sub_flux) * 3),
        label="MAD of Bkg Subtracted Flux",
    )
    ax[1].invert_yaxis()
    ax[1].set_ylabel(r"3×MAD [mag / $\rm arcsec^2$]", fontsize=14)
    # ax[1].set_xscale("log")
    ax[1].grid(True)
    ax[1].legend()

    for a in ax:
        a.set_xlabel("Radius (arcsec)", fontsize=14)

    fig.tight_layout()
    if output_path:
        plt.savefig(output_path / f"{label}_MAD_profiles.pdf")
        plt.close()
    else:
        plt.show()

In [ ]:
# | export


def measure_noise_in_annuli(
    image_path,  # path to the image file for measurement
    mask_path,  # path to the mask file
    r_max=3000,  # maximum radius in pixels
    gscale=0.4,  # geometric scale by which to grow annuli
    ellipticity=0.5,  # ellipticity to use for all annuli (1-b/a), use 0 for circles
    bkg_radius=1000,  # radius beyond which to measure background
    num_points=1000,  # number of points with which to measure noise
    pixelscale=0.3,  # pixelscale of the image in arcsec/pixel
    label=None,  # label for the output files
    output_path=None,  # path to save output
    plot_annuli=False,  # save annuli overlay plot
    save_diagnostics=False,  # save diagnostic plots
    zp=23.9,  # zero point of the image
):
    log = logging.getLogger(__name__)

    log.info(f"Requested rmax={r_max:.1f} pixels.")
    n_annuli = int(np.floor(np.log(r_max) / np.log(1 + gscale))) + 1
    r_max = (1 + gscale) ** (n_annuli)
    annuli = np.geomspace(1, r_max, n_annuli + 1)
    log.info(
        f"Measuring noise in {n_annuli} annuli with ellipticity={ellipticity:.3f}, rmax={r_max:.1f} pixels and gscale={gscale:.3f}."
    )

    image = fits.getdata(image_path)
    mask = fits.getdata(mask_path).astype(bool)
    image = np.where(mask, np.nan, image)
    mask = ~np.isfinite(image)

    image_height, image_width = image.shape
    valid_points = np.where(~mask)
    valid_points = np.column_stack((valid_points[1], valid_points[0]))
    valid_points = valid_points[
        (valid_points[:, 0] > r_max)
        & (valid_points[:, 0] < image_width - r_max)
        & (valid_points[:, 1] > r_max)
        & (valid_points[:, 1] < image_height - r_max)
    ]
    if valid_points.size < 2:
        log.error("Not enough valid points found for noise measurement.")
        return

    centres = []
    pas = []
    flux_stats = []
    np.random.seed(
        42
    )  # pick the same random points for consistency between box sizes and bands
    while len(centres) < num_points:
        log.debug(f"Centre {len(centres) + 1} of {num_points}")
        centre = valid_points[np.random.choice(len(valid_points))]
        centre = centre + np.random.uniform(-0.5, 0.5, 2)
        centre = tuple(centre)
        pa = np.random.uniform(0, 180)  # in degrees, random PA for each point
        pas.append(pa)
        centres.append(centre)
        log.debug(
            f"Extracting noise profile around point: ({centre[0]:.1f}, {centre[1]:.1f}), PA={pa:.1f}"
        )

        for i in range(len(annuli) - 1):
            sma_in, sma_out = annuli[i], annuli[i + 1]
            ba = 1.0 - ellipticity
            theta_rad = np.deg2rad(pa)
            if np.isclose(ba, 0.0):
                aperture = CircularAnnulus(centre, sma_in, sma_out)
            else:
                aperture = EllipticalAnnulus(
                    centre,
                    a_in=sma_in,
                    a_out=sma_out,
                    b_out=sma_out * ba,
                    b_in=sma_in * ba,
                    theta=theta_rad,
                ).to_mask(method="center")
            aperture_mask = aperture.to_mask(method="center")
            values = aperture_mask.get_values(image, mask)

            sma = (sma_in + sma_out) / 2
            stats = {
                "Centre_pixel": centre,
                "Inner_Radius_pix": sma_in,
                "Outer_Radius_pix": sma_out,
                "Inner_Radius_arcsec": sma_in * pixelscale,
                "Outer_Radius_arcsec": sma_out * pixelscale,
                "SMA_annulus_centre_pix": sma,
                "SMA_annulus_centre_arcsec": sma * pixelscale,
                "Ellipticity": ellipticity,
                "PA_deg": pa,
            }

            if values is None or len(values) < 2:
                stats.update(
                    {
                        key: np.nan
                        for key in [
                            "Mean_flux_annulus",
                            "Median_flux_annulus",
                            "Std_flux_annulus",
                            "Clipped_mean_flux_annulus",
                            "Clipped_median_flux_annulus",
                            "Clipped_Std_flux_annulus",
                            "Total_valid_pix_annulus",
                        ]
                    }
                )
            else:
                clipped = sigma_clip(values, sigma=3, cenfunc="median", maxiters=5)
                clipped_values = clipped.data[~clipped.mask]
                stats.update(
                    {
                        "Mean_flux_annulus": np.nanmean(values) / pixelscale**2,
                        "Median_flux_annulus": np.nanmedian(values) / pixelscale**2,
                        "Std_flux_annulus": np.nanstd(values),
                        "Clipped_mean_flux_annulus": np.nanmean(clipped_values)
                        / pixelscale**2,
                        "Clipped_median_flux_annulus": np.nanmedian(clipped_values)
                        / pixelscale**2,
                        "Clipped_Std_flux_annulus": np.nanstd(clipped_values),
                        "Total_valid_pix_annulus": len(values),
                    }
                )

            flux_stats.append(stats)

    if len(flux_stats) == 0:
        log.error("No valid points found for noise measurement.")
        return

    flux_stats = pd.DataFrame(flux_stats)

    # Background subtraction
    flux_grouped = flux_stats.groupby("Centre_pixel")
    flux_combined = []
    for centre, group in flux_grouped:
        group = group.copy()
        bkg_region = group[group["SMA_annulus_centre_pix"] > bkg_radius]
        if bkg_region.empty:
            group["bkg_sub_flux"] = group["bkg_noise"] = np.nan
        else:
            bkg = np.nanmean(bkg_region["Clipped_median_flux_annulus"])
            noise = np.nanstd(bkg_region["Clipped_median_flux_annulus"])
            group["bkg_sub_flux"] = group["Clipped_median_flux_annulus"] - bkg
            group["bkg_noise"] = 3 * noise
        flux_combined.append(group)

    extended_measurement_table = pd.concat(flux_combined, ignore_index=True)

    # Median Absolute Deviation calculation as noise
    mad_group = extended_measurement_table.groupby("SMA_annulus_centre_arcsec")
    radii = sorted(mad_group.groups.keys())
    noise_measurements = pd.DataFrame(
        {
            "SMA_annulus_centre_arcsec": radii,
            "MAD_Median_Clipped_Flux": [
                median_abs_deviation(
                    mad_group.get_group(r)["Clipped_median_flux_annulus"],
                    scale="normal",
                    nan_policy="omit",
                )
                for r in radii
            ],
            "MAD_Bkg_Subtracted_Flux": [
                median_abs_deviation(
                    mad_group.get_group(r)["bkg_sub_flux"],
                    scale="normal",
                    nan_policy="omit",
                )
                for r in radii
            ],
        }
    )

    if output_path:
        output_path = Path(output_path)
        output_path.mkdir(parents=True, exist_ok=True)
        fn = "noise_stats_extended_datatable.csv"
        if label is not None:
            fn = f"{label}_{fn}"
        extended_measurement_table.to_csv(output_path / fn, index=False)
        fn = "noise_measurements.csv"
        if label is not None:
            fn = f"{label}_{fn}"
        noise_measurements.to_csv(output_path / fn, index=False)

    if plot_annuli:
        log.info(
            "Plotting the annuli overlays on the image... May take a while if your image is big, and/or if you have lots of annuli... :) "
        )
        fig, ax = plt.subplots(figsize=(8, 8))
        ax.imshow(
            image,
            origin="lower",
            cmap="gray",
            norm=simple_norm(image, "sqrt", percent=99),
        )
        ax.set_title("Overlay of All Fitted Annuli")

        for i, coord in enumerate(centres):
            x, y = coord
            pa = pas[i]
            ba = 1.0 - ellipticity

            for j in range(len(annuli) - 1):
                sma = (annuli[j] + annuli[j + 1]) / 2
                if np.isclose(ba, 0.0):
                    # Plot circle (circular annulus)
                    ring = plt.Circle(
                        (x, y),
                        radius=sma,
                        edgecolor="magenta",
                        facecolor="none",
                        linewidth=0.4,
                        alpha=0.5,
                    )
                    ax.add_patch(ring)
                else:
                    e2 = plt.matplotlib.patches.Ellipse(
                        (x, y),
                        width=2 * sma,
                        height=2 * sma * ba,
                        angle=pa,
                        edgecolor="magenta",
                        facecolor="none",
                        linewidth=0.4,
                        alpha=0.5,
                    )
                    ax.add_patch(e2)

        ax.set_xlim(0, image.shape[1])
        ax.set_ylim(0, image.shape[0])
        plt.savefig(output_path / f"{label}_annuli_overlay.pdf")
        plt.close()

    if save_diagnostics:
        noise_profile_diagnostics(
            extended_measurement_table,
            label=label,
            output_path=output_path,
            zp=zp,
        )

    return extended_measurement_table, noise_measurements